In [1]:
import pandas as pd
import numpy as np
import json
from gensim.models.doc2vec import Doc2Vec, TaggedDocument 
from gensim.utils import simple_preprocess
from scipy.stats import pearsonr, spearmanr

In [2]:
#Functions
def convert_jsonl2df(file_path):
    # Đọc file jsonl vào DataFrame
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))
    
    df = pd.DataFrame(data)
    return df
def cosine_similarity(v1, v2):
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0:
        return 0
    return dot_product / (norm_v1 * norm_v2)
def model_evaluation(df):
    predicted_scores = []
    truth_scores = []
    for row in df.itertuples():
        words_s1 = simple_preprocess(str(row.sentence1))
        words_s2 = simple_preprocess(str(row.sentence2))
        vector_s1 = model.infer_vector(words_s1, epochs=50)
        vector_s2 = model.infer_vector(words_s2, epochs=50)
        pred_score = cosine_similarity(vector_s1, vector_s2)
        predicted_scores.append(pred_score)
        truth_scores.append(row.score)
    pearson_corr, _ = pearsonr(predicted_scores, truth_scores)
    spearman_corr, _ = spearmanr(predicted_scores, truth_scores)
    return pearson_corr, spearman_corr

In [3]:
#Dataset Preprocessing
train_path = '/kaggle/input/datasets/namtd07vn/text-sentiment-dataset/train.jsonl'
val_path = '/kaggle/input/datasets/namtd07vn/text-sentiment-dataset/validation.jsonl'
test_path = '/kaggle/input/datasets/namtd07vn/text-sentiment-dataset/test.jsonl'
train_df = convert_jsonl2df(train_path)
val_df = convert_jsonl2df(val_path)
test_df = convert_jsonl2df(test_path)
tagged_data = []
doc_id = 0
# Đọc file JSONL và trích xuất cả sentence1 và sentence2
for index, row in train_df.iterrows():
    s1 = row["sentence1"]
    s2 = row["sentence2"]
    
    # Đưa câu thứ nhất vào danh sách huấn luyện nếu câu không rỗng
    if s1:
        words_s1 = simple_preprocess(s1)
        tagged_data.append(TaggedDocument(words=words_s1, tags=[f"doc_{index}"]))
        doc_id += 1
        
    # Đưa câu thứ hai vào danh sách huấn luyện nếu câu không rỗng
    if s2:
        words_s2 = simple_preprocess(s2)
        tagged_data.append(TaggedDocument(words=words_s2, tags=[f"doc_{index}"]))
        doc_id += 1

In [4]:
#score: 0 --> 5 (integer)
# print(df['score'].describe())
# print(df[df["score"] == 0].head()) #sentence couples are irrelevant
# print(df[df["score"] == 5].head()) #sentence couples are paraphrased

# dm = 1 ---- Distributed Memory
# dm = 0 ---- Distributed Bag of Words

In [5]:
#Model Training
model = Doc2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=60, 
    dm = 0
)
# Xây dựng từ vựng và training
model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

In [6]:
#Experiment model with val dataset ---> hyperparameter tunning (dm, epochs, vector_size, min_count)
pearson_corr, spearman_corr = model_evaluation(val_df)
print(f"pearson_correlation_score: {pearson_corr:.3f}")
print(f"spearman_correlation_score: {spearman_corr:.3f}")

pearson_correlation_score: 0.492
spearman_correlation_score: 0.508


In [7]:
#evaluate model with test dataset
# model_path = "/kaggle/working/best_dov2vec_model.model"
# model = Doc2Vec.load(model_path)
pearson_corr, spearman_corr = model_evaluation(test_df)
print(f"pearson_correlation_score: {pearson_corr:.3f}")
print(f"spearman_correlation_score: {spearman_corr:.3f}")

pearson_correlation_score: 0.414
spearman_correlation_score: 0.416


In [8]:
# #save model parameters with best performance
# model_path = "/kaggle/working/best_dov2vec_model.model"
# model.save(model_path)